In [1]:
import re 
import os 
import json
from time import sleep
from io import BytesIO

from tqdm.notebook import tqdm
import pandas as pd 
from PIL import Image 
import pytesseract
from pdf2image import convert_from_bytes

import requests

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from SentimentAnalysis import sentiment_from_api, individual_sentiment_from_api
from ProminenceAnalysis import check_name_appearance
from Keyword import keyword_occurrences
#from Tier import tier_print


In [2]:
# Specify the path to tesseract executable
pytesseract.pytesseract.tesseract_cmd = '/usr/local/bin/tesseract'
# This line sets the path for the Tesseract OCR (Optical Character Recognition) engine.
# pytesseract is a Python wrapper for Google's Tesseract-OCR Engine. 
# The specified path ('/usr/local/bin/tesseract') should be the location where the 
# Tesseract binary is installed on the system.

# Lambda function used to parse the domain name from a given URL
parse_domain = lambda url: url.replace('https://', '').split('/')[0]    
# This is a lambda function, a shorthand way to define a small anonymous function in Python.
# The function 'parse_domain' takes a URL as input, removes 'https://' from it, and then 
# splits the string at the first '/' to extract the domain name.

number_regex = re.compile('(\d+)') 
# This line creates a regular expression pattern to find sequences of digits (\d+).
# The compiled pattern 'number_regex' can be used to find all occurrences of this pattern
# in a string, which is typically faster than using the pattern directly.

space_regex = re.compile(' +')
# Similarly, this line compiles a regular expression pattern to find one or more consecutive 
# spaces (' +'). It can be used for searching or manipulating strings where multiple spaces
# need to be identified or modified.

# Provide the path to the chromedriver executable
driver = webdriver.Chrome()
# This line initializes an instance of Chrome WebDriver, which is used for automating web 
# applications for testing purposes, but can also be used for other tasks like web scraping.
# This assumes that the ChromeDriver executable is in the system's PATH. If it's not, the path 
# to the ChromeDriver needs to be provided as an argument, like webdriver.Chrome('/path/to/chromedriver').


In [3]:
# The function to convert text in a list of image objects to strings using OCR
def convert_images_to_text(images):
    """
    Converts text in a list of image objects to strings using OCR.

    Parameters:
    images (list): A list of image objects to be processed.

    The function performs the following steps:
    1. Initializes an empty dictionary 'text' to hold the OCR results for each image.

    2. Iterates over the images, using enumeration to get both the image and its index:
       - Resizes each image to half its original width and height using bicubic interpolation.
       - Saves the resized image to a folder (name of the folder is not specified in the snippet).
       - Uses pytesseract to convert the image to a string. 
         Newlines are replaced with spaces for continuous text.
       - Additional formatting is applied to remove multiple spaces and non-ASCII characters.
       - The processed text is converted to lowercase and stored in the 'text' dictionary, 
         keyed by the image index.

    3. Writes the 'text' dictionary to a JSON file named 'ocr.json' in the specified folder.
       This file will contain the OCR results in a structured format.

    4. Returns the combined text from all images as a single string, joined by newlines.

    Exception Handling:
    - In case of an exception (error), the function captures it and returns the error message 
      as a string. This could be useful for debugging or logging errors in the OCR process.

    Note: 
    - The function assumes the availability of the pytesseract module and Tesseract OCR 
      installed on the system.
    - The folder in which images are saved and the JSON file is written is not defined in 
      the snippet. It should be specified by the variable 'folder_name'.
    - 'space_regex' should be a previously defined regular expression for handling spaces.
    """
    # Initialize an empty dictionary to hold the results
    text = {}

    try:
        for number, image in enumerate(images): 
            new_width = image.width // 2  # Double the width
            new_height = image.height // 2  # Double the height
            # Resize the image using bicubic interpolation
            image = image.resize((new_width, new_height), Image.BICUBIC)

            # Save the resized image and perform OCR
            image.save(f'{folder_name}/{number}.jpg')
            current_text = pytesseract.image_to_string(image).replace('\n', ' ')
            current_text = space_regex.sub(' ', current_text)
            current_text = ''.join(filter(lambda char: ord(char) <= 122, current_text))
            text[number] = current_text.lower()  # Convert to lowercase

        # Write the OCR results to a JSON file
        with open(f'{folder_name}/ocr.json', 'w') as fileout: 
            json.dump(text, fileout, indent = 4, ensure_ascii = False)

        # Return the text dictionary
        return '\n'.join(text.values())
    
    except Exception as e:
        # Handle exceptions by returning the error message
        return str(e)

In [4]:
def download_print_from_ausprint_meltwater(url, filename): 
    """
    Downloads and processes print media content from a given URL of ausprint_meltwater.

    The function aims to automate the extraction of text content from print media available 
    on ausprint_meltwater by downloading images and converting them to text using OCR.

    Parameters:
    url (str): The URL from which the print media content is to be downloaded.
    filename (str): The name of the file where the processed content will be saved.

    The function performs the following steps:
    1. Navigates to the given URL using a web driver.
    2. Extracts information about the number of pages (total and current) from a page element.
    3. Iterates over each page, capturing its content as an image. 
       - For each page, it finds the image container by ID and retrieves the image URL.
       - The image is then downloaded and converted to an RGB format Image object.
       - This Image object is added to the 'page_images' list.
    4. Once all pages are processed, it uses 'convert_images_to_text' to transcribe the images 
       to text.
    5. Extracts additional information such as the media outlet name.
    6. Loads 'tier.json' to get detailed information about the media outlet.
    7. Returns a dictionary containing the transcribed content, outlet name, and outlet info.

    Exception Handling:
    - The function includes a try-except block to handle any exceptions that might occur 
      during the process. In case of an exception, it prints an error message and returns 
      a formatted error string.

    Note: 
    - The function assumes the existence of a webdriver instance (driver), specific web 
      elements in the webpage (like 'paginator', 'clip_scan_image_', 'outlet'), and the 
      'tier.json' file for outlet information.
    - The commented-out section seems to be an alternate method for capturing the images 
      which involves clicking through pages and taking screenshots. This part is not active 
      in the current implementation.
    """
    try:
        # Attempt to navigate the web driver to the specified URL.
        page = driver.get(url)
        # Initialize an empty list to store images found on the page.
        page_images = [] 

        # Find the HTML element by class name that contains pagination information.
        element = driver.find_element(By.CLASS_NAME, "paginator")
        # Extract numbers from the element's text using a regular expression. 
        # This is typically used to find current and total page numbers.
        page_info = number_regex.findall(element.text)
        # Convert the extracted page numbers from strings to integers.
        # Assign the first number to 'current' (current page number) 
        # and the second to 'total' (total number of pages).
        current, total = int(page_info[0]), int(page_info[1]) 

#         for _ in range(total - current):
#             element = driver.find_element(By.CLASS_NAME, "right-down")
#             element_png = element.screenshot_as_png
            
#             image = Image.open(BytesIO(element_png)).convert('RGB')
#             page_images.append(image)
            
#             # Wait up to 10 seconds to ensure the button element is clickable before attempting to click it
#             button = WebDriverWait(driver, 10).until(
#                 # Filters to identify a specific interactive 'div' element acting as a "Next" button
#                 EC.element_to_be_clickable((By.XPATH, "//div[@aria-disabled='false'][@role='button'][contains(@class, 'swiper-button-next')]"))
#             )
#             # Click on the found button
#             button.click()
#             # Cause the program to pause for 2 seconds before moving on to the next iteration of the loop
#             sleep(2)

#         element = driver.find_element(By.CLASS_NAME, "right-down")
#         element_png = element.screenshot_as_png
        
#         image = Image.open(BytesIO(element_png)).convert('RGB')
#         page_images.append(image)
        
#         result = convert_images_to_text(page_images)
#         return result 

        # Initialize an empty list to store images from each page.
        page_images = []

        # Iterate over the range of total pages.
        for current in range(total):
            # Locate the image container element by its ID, which includes the current page number.
            image_container = driver.find_element(By.ID, f"clip_scan_image_{current}")
            
            # Uncomment the line below to find the 'img' tag within the container if needed.
            # image_element = image_container.find_element(By.TAG_NAME, "img")
           
            # Get the source URL of the image.
            image_url = image_container.get_attribute('src')
            # Send a request to the image URL and get the image content.
            response = requests.get(image_url).content
            # Open the image from the response, convert it to RGB format, and add it to the list.
            image = Image.open(BytesIO(response)).convert('RGB')
            page_images.append(image)

        # Convert the collected images to text (presumably through OCR or a similar process).
        result = convert_images_to_text(page_images)

        # Find the text of the 'outlet' element by its class name.
        outlet = driver.find_element(By.CLASS_NAME, 'outlet').text

        # Load information from a JSON file and match the outlet name, handling cases where it's not found.
        tier_info = json.load(open('tier.json', 'r')).get(outlet.replace('The ', ''), f'outlet [{outlet}] not found in tier.json')

        return {'content': result, 'outlet': outlet, 'outlet_info': tier_info} 
    
    except Exception as e:
        # Log the exception for debugging purposes
        # Consider logging the full stack trace for better debugging
        print(f"An error occurred: {e}")

        # Return an error message or raise a specific exception
        return f"An error occurred while processing the URL {url}: {e}"
        

In [5]:
# Define a dictionary mapping domain names to their corresponding PDF download functions.
pdf_downloader_mapping = {
    'ausprint.meltwater.com': download_print_from_ausprint_meltwater
}

# Define a function to perform OCR (Optical Character Recognition) on a print media from a URL.
def ocr_print_from_url(url, filename):
    # Extract the domain from the given URL.
    domain = parse_domain(url) 
    # Create a directory with the given filename, if it does not exist. 
    os.makedirs(filename, exist_ok = True)
    # Use the domain to find the corresponding download function from the mapping,
    # and download the PDF, returning its content as a string.
    str_content = pdf_downloader_mapping[domain](url, filename)
    
    return str_content
    

In [6]:
if __name__ == '__main__':
    
    table = pd.read_excel('Tracker2023.xlsx', skiprows = 1)
    # Initialize an empty dictionary
    record = []

    for each_index, each_url in enumerate(tqdm(table.loc[table['MEDIUM'].str.lower() == 'print']['URL'].to_list())):
        # Process each URL and store results
        filename = folder_name = f'downloads/{each_index}'
        content = ocr_print_from_url(each_url, filename) 
        
        #sentiment = sentiment_from_api(content, folder_name)

        company_name_variants = ['flybuys', 'onepass']
        appearance_data = check_name_appearance(content['content'], company_name_variants)

        keywords = ["flybuys", "onepass"]
        key_occur = keyword_occurrences(keywords, content['content'])

        content.update(appearance_data)
        
        with open(f'{filename}/result.json', 'w') as fileout:
            json.dump(content, fileout, indent = 4, ensure_ascii = False)


  0%|          | 0/12 [00:00<?, ?it/s]